# 03_features — Tabular features (per 20_Features_Model_Spec v3)

Built from **raw pkl** under the as-of rule. The label uses **gross purchase events** (leak-safe). Net M/Q uses a precomputed matched-return (weighted by purchase price). Country is computed **as of each cutoff**. Includes leakage unit tests.

Output: `processed/features_tabular.pkl`. Sequence tensors: see 03b.

> Note (F1): the label universe is **gross purchase events** (invoice existence), panel = 47,933 rows / 5,212 customers — vs the EDA net-event panel (47,813 / 5,204). Gross is more consistent with the churn target (an order later refunded still counts) and removes label-side future-return contamination entirely. See WORKLOG.


## Setup + freeze TEMPORAL_SPLIT


In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import defaultdict
from ds_common import *
INV="Invoice"; DT="InvoiceDate"; QTY="Quantity"; PR="Price"; SC="StockCode"
H=HORIZON_DAYS; LB=180

# frozen BEFORE modeling. Boundaries are EXCLUSIVE on the left window edge:
#   train = cutoffs < train_end  (2010-06 .. 2011-01)
#   val   = train_end <= cutoff < val_end  (2011-02 .. 2011-04)
#   test  = cutoff >= val_end     (2011-05 .. 2011-09)
TEMPORAL_SPLIT={"train_end":"2011-02-01","val_end":"2011-05-01","test_end":"2011-09-01"}
_te=pd.Timestamp(TEMPORAL_SPLIT["train_end"]); _ve=pd.Timestamp(TEMPORAL_SPLIT["val_end"])
def split_of(T): return "train" if T<_te else ("val" if T<_ve else "test")
print("TEMPORAL_SPLIT", TEMPORAL_SPLIT)


TEMPORAL_SPLIT {'train_end': '2011-02-01', 'val_end': '2011-05-01', 'test_end': '2011-09-01'}


## 1. Derive gross purchases / returns from raw pkl (same product filter)


In [2]:
raw=pd.read_pickle(RAW_PKL)
raw=raw.drop_duplicates(subset=[c for c in raw.columns if c!="source_sheet"]).copy()   # raw is NOT deduped
raw[DT]=raw[DT].astype("datetime64[ns]")
inv=raw[INV].astype(str); prod=raw[SC].astype(str).str.match(r"^\d{5}"); canc=inv.str.startswith("C")
gross=raw[(~canc)&prod&(raw[QTY]>0)&(raw[PR]>0)&raw[KEY].notna()].copy(); gross[KEY]=gross[KEY].astype("int64")
ret  =raw[(canc)&prod&raw[KEY].notna()&(raw[QTY]<0)&(raw[PR]>0)].copy()   # explicit return filter
ret[KEY]=ret[KEY].astype("int64"); ret["ret_qty"]=ret[QTY].abs()
assert (ret[QTY]<0).all() and (ret[PR]>0).all(), "return filter"
gross=gross.reset_index(drop=True); gross["lid"]=np.arange(len(gross)); gross["line_val"]=gross[QTY]*gross[PR]
gross_sorted=gross.sort_values(DT)                                        # for as-of country (.last())
print("gross lines",len(gross),"| return lines",len(ret))


gross lines 776577 | return lines 17586


## 2. Precompute matched-return (time-constrained, purchase-price weighted) + per-invoice fully_returned_date


In [3]:
purch=defaultdict(list)
for lid,k,s,d,q,p in zip(gross["lid"],gross[KEY],gross[SC],gross[DT],gross[QTY],gross[PR]):
    purch[(k,s)].append([d,lid,float(q),float(p)])   # date, lid, remaining, price
rets=defaultdict(list)
for k,s,d,q in zip(ret[KEY],ret[SC],ret[DT],ret["ret_qty"]): rets[(k,s)].append((d,float(q)))
line_zero={}; rrecs=[]
for (k,s),rl in rets.items():
    plist=purch.get((k,s))
    for rdate,rq in sorted(rl):
        mval=0.0; mqty=0.0; rem=rq
        if plist:
            for row in sorted([r for r in plist if r[0]<rdate], key=lambda r:r[0], reverse=True):  # prior only, LIFO
                if rem<=0: break
                take=min(row[2],rem); row[2]-=take; rem-=take; mqty+=take; mval+=take*row[3]   # purchase price
                if row[2]<=0 and row[1] not in line_zero: line_zero[row[1]]=rdate
        rrecs.append((k,rdate,mval,mqty,rem))
R=pd.DataFrame(rrecs,columns=[KEY,"rdate","matched_val","matched_qty","unmatched_qty"])
print("matched-return rows",len(R),"| unmatched qty",int(R["unmatched_qty"].sum()))


matched-return rows 17586 | unmatched qty 14245


## 3. Gross purchase events (invoice level) + fully_returned_date


In [4]:
gl=gross.copy(); gl["zero"]=gl["lid"].map(line_zero)
ev=(gl.groupby([KEY,INV]).agg(event_time=(DT,"min"),gqty=(QTY,"sum"),gval=("line_val","sum"),
        n_items=(SC,"nunique"),nlines=("lid","size"),nzeroed=("zero",lambda z:z.notna().sum()),
        frmax=("zero","max")).reset_index())
ev["fully_returned_date"]=pd.to_datetime(np.where(ev["nzeroed"]==ev["nlines"],ev["frmax"],pd.NaT))
ev=ev.sort_values([KEY,"event_time"]).reset_index(drop=True)
maxd=ev["event_time"].max()
print("gross invoice events",len(ev),"| customers",ev[KEY].nunique(),"| maxd",maxd.date())


gross invoice events 36594 | customers 5852 | maxd 2011-12-09


## 4. Build as-of features + label per cutoff (country computed as-of)


In [5]:
cuts=pd.date_range("2010-06-01",maxd-pd.Timedelta(days=H),freq="MS")
assert cuts.max()<=pd.Timestamp(TEMPORAL_SPLIT["test_end"]), "cutoffs exceed test_end"
evk=ev[[KEY,"event_time","gqty","gval","n_items","fully_returned_date"]]
rows=[]
for T in cuts:
    e=evk[evk["event_time"]<=T]
    ar=e[e["event_time"]>T-pd.Timedelta(days=LB)][KEY].unique()                     # at-risk population
    fut=set(evk[(evk["event_time"]>T)&(evk["event_time"]<=T+pd.Timedelta(days=H))][KEY].unique())
    e=e[e[KEY].isin(ar)]; g=e.groupby(KEY)
    f=pd.DataFrame(index=pd.Index(ar,name=KEY))
    f["recency_days"]=(T-g["event_time"].max()).dt.days
    f["tenure_days"]=(T-g["event_time"].min()).dt.days
    f["frequency_total"]=g.size()
    f["gross_monetary_total"]=g["gval"].sum(); f["gross_quantity_total"]=g["gqty"].sum()
    f["avg_n_items"]=g["n_items"].mean()
    for w in [30,90,180]:
        ew=e[e["event_time"]>T-pd.Timedelta(days=w)].groupby(KEY)
        f[f"freq_{w}d"]=ew.size().reindex(ar).fillna(0)
        f[f"gross_monetary_{w}d"]=ew["gval"].sum().reindex(ar).fillna(0)
    rec=e[e["event_time"]>T-pd.Timedelta(days=90)].groupby(KEY)
    pri=e[(e["event_time"]<=T-pd.Timedelta(days=90))&(e["event_time"]>T-pd.Timedelta(days=180))].groupby(KEY)
    rf=rec.size().reindex(ar).fillna(0); pf=pri.size().reindex(ar).fillna(0)
    rm=rec["gval"].sum().reindex(ar).fillna(0); pm=pri["gval"].sum().reindex(ar).fillna(0)
    f["trend_freq_90v90_logdiff"]=np.log1p(rf)-np.log1p(pf)                          # unified: log1p difference
    f["trend_monetary_90v90_logdiff"]=np.log1p(rm)-np.log1p(pm)
    f["previous_90d_zero"]=(pf==0).astype(int)
    rr=R[R["rdate"]<=T].groupby(KEY).agg(matched_val=("matched_val","sum"),matched_qty=("matched_qty","sum"),
            unmatched_qty=("unmatched_qty","sum"),return_count=("rdate","size"),last_ret=("rdate","max"))
    f=f.join(rr)
    for c in ["matched_val","matched_qty","unmatched_qty","return_count"]: f[c]=f[c].fillna(0)
    nm=f["gross_monetary_total"]-f["matched_val"]; nq=f["gross_quantity_total"]-f["matched_qty"]
    assert (nm>=-1e-6).all() and (nq>=-1e-6).all(), "net < gross violated"   # detect errors before clipping fp noise
    f["net_monetary_total"]=nm.clip(lower=0); f["net_quantity_total"]=nq.clip(lower=0)
    f["return_rate_value"]=np.where(f["gross_monetary_total"]>0,f["matched_val"]/f["gross_monetary_total"],0.0)
    f["has_unmatched_returns"]=(f["unmatched_qty"]>0).astype(int)          # renamed from over_return_flag
    f["has_return_history"]=(f["return_count"]>0).astype(int)
    f["days_since_last_return"]=(T-f["last_ret"]).dt.days
    fr=e[e["fully_returned_date"].notna()&(e["fully_returned_date"]<=T)].groupby(KEY).size()
    f["fully_returned_order_count"]=fr.reindex(ar).fillna(0)
    f["fully_returned_order_rate"]=f["fully_returned_order_count"]/f["frequency_total"]
    f["is_single_purchase_history"]=(f["frequency_total"]==1).astype(int)   # history count == frequency_total (not duplicated)
    mth=T.month; f["month_sin"]=np.sin(2*np.pi*mth/12); f["month_cos"]=np.cos(2*np.pi*mth/12)
    # country as-of cutoff (most recent observed country <= T); keep only country + is_uk (grouping done in 04, train-fit)
    gp=gross_sorted[gross_sorted[DT]<=T]                                          # gross lines strictly <= T
    casof=gp.groupby(KEY)["Country"].last().reindex(ar)
    f["country"]=casof.values; f["is_uk"]=(f["country"]=="United Kingdom").astype(int)
    f["max_purchase_time_used"]=g["event_time"].max(); f["max_return_time_used"]=f["last_ret"]
    f["cutoff"]=T; f["label_window_end"]=T+pd.Timedelta(days=H); f["split"]=split_of(T); f["churn"]=(~f.index.isin(fut)).astype(int)
    rows.append(f.drop(columns=["last_ret"]).reset_index())
panel=pd.concat(rows,ignore_index=True)
print("PANEL",panel.shape,"| customers",panel[KEY].nunique(),"| churn%",round(panel["churn"].mean()*100,1))
print("split rows:",panel["split"].value_counts().to_dict())
print("churn by split:",panel.groupby("split")["churn"].mean().round(3).to_dict())


PANEL (47933, 39) | customers 5212 | churn% 50.9
split rows: {'train': 23987, 'test': 13961, 'val': 9985}
churn by split: {'test': 0.469, 'train': 0.502, 'val': 0.58}


## 5. Leakage unit tests (spec §21)


In [6]:
assert (panel["max_purchase_time_used"]<=panel["cutoff"]).all(), "purchase after cutoff!"
assert (panel["max_return_time_used"].isna()|(panel["max_return_time_used"]<=panel["cutoff"])).all(), "return after cutoff!"
assert (panel["net_monetary_total"]<=panel["gross_monetary_total"]+1e-6).all(), "net>gross!"
assert (panel["recency_days"]>=0).all() and (panel["tenure_days"]>=panel["recency_days"]).all()
assert (panel["label_window_end"]==panel["cutoff"]+pd.Timedelta(days=H)).all()
assert panel[[KEY,"cutoff"]].duplicated().sum()==0
# diverse-implementation check: rebuild one cutoff's at-risk & churn straight from events, assert equal
T0=pd.Timestamp("2011-03-01")
ar0=evk[(evk["event_time"]>T0-pd.Timedelta(days=LB))&(evk["event_time"]<=T0)][KEY].unique()
fut0=set(evk[(evk["event_time"]>T0)&(evk["event_time"]<=T0+pd.Timedelta(days=H))][KEY].unique())
p0=panel[panel["cutoff"]==T0]
assert set(p0[KEY])==set(ar0) and p0["churn"].sum()==sum(1 for c in ar0 if c not in fut0), "cutoff rebuild mismatch"
print("leakage tests PASSED:",len(panel),"rows")


leakage tests PASSED: 47933 rows


## 6. Save + feature list


In [7]:
panel.to_pickle(PROC_DIR/"features_tabular.pkl")
meta=[KEY,"cutoff","label_window_end","split","churn","country","max_purchase_time_used","max_return_time_used"]
feat=[c for c in panel.columns if c not in meta]
print("saved features_tabular.pkl", panel.shape)
print("n features:",len(feat)); print(feat)
panel.head()


saved features_tabular.pkl (47933, 39)
n features: 31
['recency_days', 'tenure_days', 'frequency_total', 'gross_monetary_total', 'gross_quantity_total', 'avg_n_items', 'freq_30d', 'gross_monetary_30d', 'freq_90d', 'gross_monetary_90d', 'freq_180d', 'gross_monetary_180d', 'trend_freq_90v90_logdiff', 'trend_monetary_90v90_logdiff', 'previous_90d_zero', 'matched_val', 'matched_qty', 'unmatched_qty', 'return_count', 'net_monetary_total', 'net_quantity_total', 'return_rate_value', 'has_unmatched_returns', 'has_return_history', 'days_since_last_return', 'fully_returned_order_count', 'fully_returned_order_rate', 'is_single_purchase_history', 'month_sin', 'month_cos', 'is_uk']


,Customer ID,recency_days,tenure_days,frequency_total,gross_monetary_total,gross_quantity_total,avg_n_items,freq_30d,gross_monetary_30d,freq_90d,...,month_sin,month_cos,country,is_uk,max_purchase_time_used,max_return_time_used,cutoff,label_window_end,split,churn
0,12346,90,90,1,27.05,5,5.0,0.0,0.00,0.0,...,1.224647e-16,-1.0,United Kingdom,1,2010-03-02 13:08:00,NaT,2010-06-01,2010-08-30,train,0
1,12349,32,32,1,1068.52,473,46.0,0.0,0.00,1.0,...,1.224647e-16,-1.0,Italy,0,2010-04-29 13:20:00,2009-12-04 12:49:00,2010-06-01,2010-08-30,train,1
2,12355,10,10,1,488.21,303,22.0,1.0,488.21,1.0,...,1.224647e-16,-1.0,Bahrain,0,2010-05-21 11:59:00,NaT,2010-06-01,2010-08-30,train,1
3,12358,174,174,1,1429.83,309,17.0,0.0,0.00,0.0,...,1.224647e-16,-1.0,Austria,0,2009-12-08 07:59:00,NaT,2010-06-01,2010-08-30,train,0
4,12359,69,177,4,1522.23,777,16.5,0.0,0.00,2.0,...,1.224647e-16,-1.0,Cyprus,0,2010-03-23 08:55:00,2010-04-12 08:40:00,2010-06-01,2010-08-30,train,0


## Feature dictionary

| feature | definition | gross/net |
|---|---|---|
| recency_days | T − most recent **gross purchase** event | gross |
| frequency_total / freq_{30,90,180}d | gross purchase-event count (total / window) | gross |
| gross_monetary_total / _{w}d | gross purchase value (total / window) | gross |
| net_monetary_total / net_quantity_total | gross − matched(return≤T); asserted ≥ gross before clip | **net as-of** |
| trend_freq_90v90_logdiff / trend_monetary_90v90_logdiff | log1p(last 90) − log1p(prior 90); previous_90d_zero flag | gross |
| return_count / return_rate_value | matched_val/gross_monetary; unmatched kept separate | matched (purchase price) |
| has_unmatched_returns | returns with no prior purchase to match (mostly left-censoring, NOT over-return) | — |
| fully_returned_order_count / _rate | invoice with all lines net 0 (definition C); rate = /frequency | net |
| is_single_purchase_history | single-purchase flag (history count == frequency_total, not stored separately) | gross |
| month_sin / month_cos | cyclical encoding of the cutoff month | — |
| country / is_uk | **as-of** most recent country ≤ T (top-k grouping deferred to 04, train-fit) | — |
| cutoff / label_window_end / max_*_time_used | audit columns (asserted ≤ cutoff) | — |


